In [25]:
import os
import numpy as np
import pandas as pd
import math
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [26]:
# Tạo thư mục lưu kết quả và model
output_dir = '../../../'  # This points to the parent folder
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

output_dir_result = os.path.join(output_dir, 'Result/AAPL/ARIMA/')
if not os.path.exists(output_dir_result):
    os.makedirs(output_dir_result, exist_ok=True)

output_dir_models = os.path.join(output_dir, 'Models/AAPL/ARIMA/')
if not os.path.exists(output_dir_models):
    os.makedirs(output_dir_models, exist_ok=True)

In [27]:
# Đường dẫn tới tệp CSV
file_path = '../../../Data/AAPL.csv'

# Đọc CSV, bỏ qua 2 hàng đầu (Ticker và Date NaN)
df = pd.read_csv(file_path, skiprows=2)

# Đặt tên cột thủ công (dựa trên cấu trúc file)
df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

# Chuyển Date thành datetime và set làm index
df['Date'] = pd.to_datetime(df['Date'], format='mixed')  # Format ngày tháng kiểu MM/DD/YYYY
df.set_index('Date', inplace=True)

# Kiểm tra DataFrame sau xử lý (tùy chọn)
print(df.head())  # Nên hiển thị dữ liệu từ 2014-09-17
print(df.tail())  # Kiểm tra ngày cuối
df.info()  # Xem số hàng và kiểu dữ liệu

               Close      High       Low      Open     Volume
Date                                                         
1980-12-12  0.098597  0.099025  0.098597  0.098597  469033600
1980-12-15  0.093453  0.093881  0.093453  0.093881  175884800
1980-12-16  0.086594  0.087022  0.086594  0.087022  105728000
1980-12-17  0.088737  0.089165  0.088737  0.088737   86441600
1980-12-18  0.091309  0.091738  0.091309  0.091309   73449600
                 Close        High         Low        Open    Volume
Date                                                                
2025-07-17  210.020004  211.800003  209.589996  210.570007  48068100
2025-07-18  211.179993  211.789993  209.699997  210.869995  48974600
2025-07-21  212.479996  215.779999  211.630005  212.100006  51377400
2025-07-22  214.399994  214.949997  212.229996  213.139999  46404100
2025-07-23  214.149994  215.149994  212.410004  215.000000  46913800
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 11243 entries, 1980-12-12 to 2

In [28]:
# Tạo biểu đồ tương tác với Plotly
fig = px.line(df, x=df.index, y='Close', 
              title='Full AAPL Close Prices', 
              labels={'Close': 'Close Price (USD)', 'Date': 'Date'})

# Tùy chỉnh để đẹp hơn và đầy đủ hơn
fig.update_layout(
    template='plotly_white',  # Theme trắng sáng, chuyên nghiệp (có thể thay bằng 'plotly_dark' nếu thích)
    xaxis_title='Date',
    yaxis_title='Close Price (USD)',
    font=dict(size=12),  # Font chữ lớn hơn
    xaxis=dict(
        tickformat='%Y-%m-%d',  # Định dạng ngày tháng
        tickangle=-45,  # Rotate nhãn trục x
        rangeselector=dict(  # Thêm nút chọn khoảng thời gian (1m, 6m, YTD, All)
            buttons=list([
                dict(count=1, label='1m', step='month', stepmode='backward'),
                dict(count=6, label='6m', step='month', stepmode='backward'),
                dict(count=1, label='YTD', step='year', stepmode='todate'),
                dict(step='all')
            ])
        ),
        type='date'  # Đảm bảo trục x là kiểu date
    ),
    yaxis=dict(gridcolor='LightGray'),  # Grid nhẹ nhàng
    hovermode='x unified',  # Hover hiển thị thống nhất
    showlegend=True,  # Hiển thị legend
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)  # Legend ngang phía trên
)

# Save plot
fig.write_html(os.path.join(output_dir_result, 'arima_full_AAPL_Close_prices.html'))

# Hiển thị biểu đồ (trong Jupyter Notebook sẽ tự render)
fig.show()

In [29]:
# Split train/test với 90% train
to_row = int(len(df) * 0.9)
training_data = df[0:to_row]['Close']
testing_data = df[to_row:]['Close']
training_data, testing_data

(Date
 1980-12-12      0.098597
 1980-12-15      0.093453
 1980-12-16      0.086594
 1980-12-17      0.088737
 1980-12-18      0.091309
                  ...    
 2021-01-22    135.663055
 2021-01-25    139.418686
 2021-01-26    139.652832
 2021-01-27    138.579788
 2021-01-28    133.731567
 Name: Close, Length: 10118, dtype: float64,
 Date
 2021-01-29    128.727219
 2021-02-01    130.853821
 2021-02-02    131.682968
 2021-02-03    130.658691
 2021-02-04    134.024216
                  ...    
 2025-07-17    210.020004
 2025-07-18    211.179993
 2025-07-21    212.479996
 2025-07-22    214.399994
 2025-07-23    214.149994
 Name: Close, Length: 1125, dtype: float64)

In [30]:
# Vẽ plot train/test split
fig = go.Figure()

# Thêm training data
fig.add_trace(go.Scatter(
    x=training_data.index,
    y=training_data.values,
    mode='lines',
    name='Training Data (90%)',
    line=dict(color='blue', width=1.5)
))

# Thêm testing data
fig.add_trace(go.Scatter(
    x=testing_data.index,
    y=testing_data.values,
    mode='lines',
    name='Testing Data (10%)',
    line=dict(color='red', width=1.5)
))

# Thêm đường phân chia - fix lỗi
split_date = testing_data.index[0]
fig.add_shape(
    type="line",
    x0=split_date, x1=split_date,
    y0=0, y1=1,
    yref="paper",
    line=dict(color="green", width=2, dash="dash")
)

# Thêm annotation cho đường phân chia
fig.add_annotation(
    x=split_date,
    y=0.9,
    yref="paper",
    text="Train/Test Split",
    showarrow=False,
    bgcolor="white",
    bordercolor="green"
)

# Cập nhật layout
fig.update_layout(
    title='AAPL Stock Price - Train/Test Split (90/10)',
    xaxis_title='Date',
    yaxis_title='Close Price ($)',
    template='plotly_white',
    width=1000,
    height=500
)

# Save plot
# fig.write_image(os.path.join(output_dir_result, 'arima_train_test_split.png'), 
#                 width=1000, height=500, scale=3)
fig.write_html(os.path.join(output_dir_result, 'arima_train_test_split.html'))


fig.show()


In [31]:
# Rolling forecast
model_predictions = []
n_test_obser = len(testing_data)

for i in range(n_test_obser):
    model = ARIMA(training_data, order=(4,1,0))
    model_fit = model.fit()
    output = model_fit.forecast()
    yhat = output.iloc[0]
    model_predictions.append(yhat)
    actual_test_value = testing_data.iloc[i]
    training_data = pd.concat([training_data, pd.Series([actual_test_value], index=[testing_data.index[i]])])
    print(output)

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index be

10118    134.100845
dtype: float64


c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so wil

10119    128.984733
dtype: float64


c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so wil

...

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so wil

11240    211.108574
dtype: float64


c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so wil

11241    212.481817
dtype: float64
11242    214.382262
dtype: float64


c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

c:\Users\VincentPham\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.



In [34]:
# Vẽ plot prediction với Plotly
fig = go.Figure()

data_range = df[to_row:].index

# Thêm predicted prices
fig.add_trace(go.Scatter(
    x=data_range,
    y=model_predictions,
    mode='lines+markers',
    name='ARIMA BTC Predicted prices',
    line=dict(color='blue', dash='dash', width=2),
    marker=dict(size=4, color='blue')
))

# Thêm actual prices
fig.add_trace(go.Scatter(
    x=data_range,
    y=testing_data,
    mode='lines',
    name='BTC Actual prices',
    line=dict(color='red', width=2)
))

# Cập nhật layout
fig.update_layout(
    title='ARIMA BTC Price Prediction',
    xaxis_title='Date',
    yaxis_title='Price',
    template='plotly_white',
    width=1000,
    height=500,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Thêm grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Save plot vào output_dir_result
fig.write_html(os.path.join(output_dir_result, 'arima_price_prediction.html'))

# Nếu muốn save PNG (cần kaleido):
# fig.write_image(os.path.join(output_dir_result, 'arima_btc_price_prediction.png'), 
#                 width=1500, height=900, scale=2)

fig.show()


In [35]:
# Tính residuals
residuals = testing_data - model_predictions

# Vẽ residual plot với Plotly
fig = go.Figure()

# Thêm residuals
fig.add_trace(go.Scatter(
    x=data_range,
    y=residuals,
    mode='lines',
    name='ARIMA Residuals',
    line=dict(color='green', width=2)
))

# Thêm đường zero line để dễ nhìn
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

# Cập nhật layout
fig.update_layout(
    title='ARIMA Residual Plot for BTC Prediction',
    xaxis_title='Date',
    yaxis_title='Residual',
    template='plotly_white',
    width=1000,
    height=500,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Thêm grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Save plot vào output_dir_result
fig.write_html(os.path.join(output_dir_result, 'arima_residual_plot.html'))

# Nếu muốn save PNG (cần kaleido):
# fig.write_image(os.path.join(output_dir_result, 'arima_btc_residual_plot.png'), 
#                 width=1500, height=900, scale=2)

fig.show()


In [36]:
from sklearn.metrics import r2_score

# Tính metrics
mape = np.mean(np.abs(testing_data - model_predictions) / np.abs(testing_data))  # MAPE thủ công
mae = mean_absolute_error(testing_data, model_predictions)
mse = mean_squared_error(testing_data, model_predictions)
rmse = math.sqrt(mse)
r2 = r2_score(testing_data, model_predictions)  # Thêm R2 Score

# Tạo DataFrame so sánh
results_df = pd.DataFrame({
    'Date': data_range,
    'Predicted': model_predictions,
    'Actual': testing_data
})

# Thêm metrics vào DataFrame
results_df['MAPE'] = mape
results_df['MAE'] = mae
results_df['MSE'] = mse
results_df['RMSE'] = rmse
results_df['R2_Score'] = r2  # Thêm R2 Score

# Save CSV vào output_dir_result
results_df.to_csv(os.path.join(output_dir_result, 'arima_predictions_and_metrics.csv'), index=False)

# In metrics để kiểm tra
print('ARIMA Mean Absolute Percentage Error:', mape)
print('ARIMA Mean Squared Error:', mse)
print('ARIMA Root Mean Squared Error:', rmse)
print('ARIMA Mean Absolute Error:', mae)
print('ARIMA R2 Score:', r2)  # In R2 Score


ARIMA Mean Absolute Percentage Error: 0.012760319262145712
ARIMA Mean Squared Error: 9.458586374156193
ARIMA Root Mean Squared Error: 3.0754814865572175
ARIMA Mean Absolute Error: 2.1714147211773334
ARIMA R2 Score: 0.9917095155728041


In [37]:
# Save model vào output_dir_models (lưu model_fit cuối cùng từ loop)
joblib.dump(model_fit, os.path.join(output_dir_models, 'AAPL_arima_model.pkl'))
print("ARIMA model saved successfully.")

ARIMA model saved successfully.
